# Synthetic Data Exploration And Online Training

Ce notebook explore le dataset synthetique annuel et entraine un premier modele incremental leger.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

DATASET_PATH = Path("../data/synthetic/synthetic_year.csv")
DATASET_PATH

: 

In [ ]:
df = pd.read_csv(DATASET_PATH)
df.head()

In [ ]:
df.shape, df.columns.tolist()

## Quick Checks

In [ ]:
display(df.describe(include="all").T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sns.histplot(df["temp_air_c"], kde=True, ax=axes[0, 0])
sns.histplot(df["hr_pct"], kde=True, ax=axes[0, 1])
sns.histplot(df["solar_wm2"], kde=True, ax=axes[1, 0])
sns.histplot(df["soc_battery_pct"], kde=True, ax=axes[1, 1])
axes[0, 0].set_title("Air Temperature")
axes[0, 1].set_title("Relative Humidity")
axes[1, 0].set_title("Solar Radiation")
axes[1, 1].set_title("Battery SOC")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x="vcrc_state", ax=axes[0])
sns.countplot(data=df, x="sorbent_mode", order=sorted(df["sorbent_mode"].unique()), ax=axes[1])
sns.countplot(data=df, x="heater_on", ax=axes[2])
axes[0].set_title("VCRC Labels")
axes[1].set_title("Sorbent Labels")
axes[2].set_title("Heater Labels")
plt.tight_layout()

## Feature Matrix

In [ ]:
feature_columns = [
    "hour",
    "temp_air_c",
    "hr_pct",
    "solar_wm2",
    "pv_voltage",
    "temp_collector_c",
    "temp_cond_c",
    "delta_hr_sorbent",
    "reservoir_level_pct",
    "soc_battery_pct",
    "dew_point_c",
    "humidity_ratio_gkg",
]

X = df[feature_columns]
y_vcrc = df["vcrc_state"]
y_sorbent = df["sorbent_mode"]
y_heater = df["heater_on"].astype(int)
X.head()

## First Incremental Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_vcrc,
    test_size=0.2,
    random_state=42,
    stratify=y_vcrc,
)

vcrc_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", random_state=42),
)

vcrc_model.fit(X_train, y_train)
y_pred = vcrc_model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("VCRC Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

## Incremental Update Sketch

In [ ]:
stream_batch = X_test.iloc[:32]
stream_labels = y_test.iloc[:32]

online_model = SGDClassifier(loss="log_loss", random_state=42)
scaler = StandardScaler()

X_bootstrap = scaler.fit_transform(X_train.iloc[:512])
online_model.partial_fit(X_bootstrap, y_train.iloc[:512], classes=[0, 1])

X_stream = scaler.transform(stream_batch)
online_model.partial_fit(X_stream, stream_labels)
online_model.coef_, online_model.intercept_